# Camada SILVER

**Objetivo desta camada:** limpeza e padronização, deixar o dado confiável e consultável.

In [0]:
CATALOGO = "workspace"

## 1. Mapeamento de colunas

In [0]:
MAPA_COLUNAS = {
    "periodo":       "Período",                 # "YYYY/MM"
    "campo":         "Campo",
    "operadora":     "Operadora atual",         
    "poco":          "Poço ANP",
    "bacia":         "Bacia",
    "localizacao":   "Localização",             # TERRA / MAR
    "instalacao":    "Nome da Instalação",      
    "ambiente":      "Ambiente",                # Produção pré-sal / Produção pós-sal mar / Produção terra
    "petroleo":      "Óleo (bbl/d)",
    "gas":           "Gás Natural (mil m3/d)",
    "condensado":    "Condensado (bbl/d)",      
    "agua":          "Água (bbl/d)",
}

## 2. Ler camada Bronze e renomear colunas para o padrão criado

In [0]:
from pyspark.sql import functions as F

df = spark.table(f"{CATALOGO}.bronze.producao_poco")

colunas_existentes = set(df.columns)
for novo, original in MAPA_COLUNAS.items():
    if original in colunas_existentes:
        df = df.withColumnRenamed(original, novo)
    else:
        print(f"AVISO: coluna '{original}' não encontrada. Ajuste o MAPA_COLUNAS.")

## 3. Transformações de limpeza

In [0]:
def texto_para_numero(coluna):
    # Converte texto numérico para double. Trata vírgula decimal se aparecer.
    limpo = F.regexp_replace(F.col(coluna), ",", ".")   # segurança: vírgula -> ponto
    return limpo.cast("double")

df_silver = df

# 3.1 Converte volumes de texto para número
for vol in ["petroleo", "gas", "condensado", "agua"]:
    if vol in df_silver.columns:
        df_silver = df_silver.withColumn(vol, texto_para_numero(vol))

# 3.2 Padroniza texto: tira espaços, acentos e uniformiza as letras maiúsculas
for txt in ["campo", "bacia", "operadora", "instalacao", "localizacao", "ambiente"]:
    if txt in df_silver.columns:
        df_silver = df_silver.withColumn(txt, F.upper(F.trim(F.col(txt))))

# 3.3 Padroniza localização para TERRA ou MAR
if "localizacao" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "localizacao",
        F.when(F.col("localizacao").rlike("MAR|OFFSHORE|OFF"), F.lit("MAR"))
         .when(F.col("localizacao").rlike("TERRA|ONSHORE|ON"), F.lit("TERRA"))
         .otherwise(F.col("localizacao")))

# 3.3b Padroniza o ambiente: "Produção pré-sal", "Produção pós-sal mar", "Produção terra" -> PRE-SAL / POS-SAL / TERRA.
if "ambiente" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "classificacao",
        F.when(F.col("ambiente").rlike("PR[EÉ]"), F.lit("PRE-SAL"))
         .when(F.col("ambiente").rlike("P[OÓ]S"), F.lit("POS-SAL"))
         .when(F.col("ambiente").rlike("TERRA"), F.lit("TERRA"))
         .otherwise(F.col("ambiente")))

# 3.4 Data de referência a partir do período (formato "2021/01" = YYYY/MM)
if "periodo" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "data_ref",
        F.coalesce(
            F.try_to_date("periodo", "yyyy/MM"),
            F.try_to_date("periodo", "MM/yyyy"),
            F.try_to_date("periodo", "yyyy-MM")))
    df_silver = (df_silver
        .withColumn("ano",  F.year("data_ref"))
        .withColumn("mes",  F.month("data_ref")))

# 3.5 Nulos de volume viram 0
for vol in ["petroleo", "gas", "condensado", "agua"]:
    if vol in df_silver.columns:
        df_silver = df_silver.fillna(0, subset=[vol])

# 3.6 Remove duplicatas exatas
antes = df_silver.count()
df_silver = df_silver.dropDuplicates()
depois = df_silver.count()
print(f"Duplicatas removidas: {antes - depois:,}")

Duplicatas removidas: 0


## 4. Gravar a camada Silver

In [0]:
(df_silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.silver.producao_poco"))

print("Tabela silver.producao_poco gravada.")
display(spark.sql(f"SELECT * FROM {CATALOGO}.silver.producao_poco LIMIT 10"))

Tabela silver.producao_poco gravada.


periodo,campo,operadora,poco,bacia,localizacao,instalacao,ambiente,petroleo,gas,condensado,agua,classificacao,data_ref,ano,mes
2025/10,MIRANGA,PETRORECÔNCAVO,7-MG-352D-BA,RECÔNCAVO,TERRA,EST. B DE MIRANGA,PRODUÇÃO TERRA,57.9346283216129,4.819903225806452,0.0,2140.023447307742,TERRA,2025-10-01,2025,10
2025/10,MIRANGA,PETRORECÔNCAVO,7-MG-402-BA,RECÔNCAVO,TERRA,EST. A DE MIRANGA,PRODUÇÃO TERRA,3.218962440322581,1.2267741935483871,0.0,5.363787650322581,TERRA,2025-10-01,2025,10
2025/10,MIRANGA,PETRORECÔNCAVO,7-MG-453D-BA,RECÔNCAVO,TERRA,EST. B DE MIRANGA,PRODUÇÃO TERRA,21.65865638935484,0.05719354838709677,0.0,23.50745473516129,TERRA,2025-10-01,2025,10
2025/10,MIRANGA,PETRORECÔNCAVO,7-MGP-72D-BA,RECÔNCAVO,TERRA,EST. D DE MIRANGA,PRODUÇÃO TERRA,0.0,2.71158064516129,0.0068985012903225815,6.837429264193548,TERRA,2025-10-01,2025,10
2025/10,MIRANGA,PETRORECÔNCAVO,7-MGP-90D-BA,RECÔNCAVO,TERRA,EST. D DE MIRANGA,PRODUÇÃO TERRA,0.0,9.336677419354839,0.2739110806451613,5.6334378919354835,TERRA,2025-10-01,2025,10
2025/10,MONTE ALEGRE,3R POTIGUAR,7-MAG-104D-RN,POTIGUAR,TERRA,ESTAÇÃO CENTRAL DE MONTE ALEGRE,PRODUÇÃO TERRA,6.645285713548388,0.008258064516129033,0.0,448.42267068354846,TERRA,2025-10-01,2025,10
2025/10,MONTE ALEGRE,3R POTIGUAR,7-MAG-10-RN,POTIGUAR,TERRA,ESTAÇÃO CENTRAL DE MONTE ALEGRE,PRODUÇÃO TERRA,0.0,0.0,0.0,0.0,TERRA,2025-10-01,2025,10
2025/10,MONTE ALEGRE,3R POTIGUAR,7-MAG-30-RN,POTIGUAR,TERRA,ESTAÇÃO CENTRAL DE MONTE ALEGRE,PRODUÇÃO TERRA,4.385620746774193,0.006870967741935484,0.0,82.01486156096773,TERRA,2025-10-01,2025,10
2025/10,MONTE ALEGRE,3R POTIGUAR,7-MAG-78-RN,POTIGUAR,TERRA,ESTAÇÃO CENTRAL DE MONTE ALEGRE,PRODUÇÃO TERRA,0.0,0.0,0.0,0.0,TERRA,2025-10-01,2025,10
2025/10,MONTE ALEGRE,3R POTIGUAR,7-MAG-93D-RN,POTIGUAR,TERRA,ESTAÇÃO CENTRAL DE MONTE ALEGRE,PRODUÇÃO TERRA,0.0,0.0,0.0,0.0,TERRA,2025-10-01,2025,10
